# Alerts-list widget: raster scan vs a global alert point index

The curated alerts-list widget lists the integrated alerts in an AOI, **newest first**, each row with
**lat, lon, alert date and intersection info**, filterable by **date range counted back from the latest
data update** (last day / 3 days / week), **confidence** and **intersection** (e.g. natural lands), with a
row limit / paging for AOIs with many alerts. This notebook compares, for AOIs from a small municipality
to a whole country:

- **Raster (per request):** read the alerts and natural lands zarr chunks under the AOI on the fly, filter,
  test pixel centers against the AOI (`shapely.contains_xy`), sort newest first. Runs in one process with a
  thread pool over tiles, like an API worker.
- **Vector (precomputed):** once per alerts release, a Dask cluster extracts the last week's alert pixels
  worldwide with their natural lands class into one compact parquet (`row`, `col`, `alert_date`,
  `confidence`, `natural_lands_class`; about 5 bytes per point). A request selects row groups from the
  footer statistics (cached), keeps those whose box intersects the AOI polygon, reads them in one
  pre-buffered read, and runs the point-in-polygon test only for row groups on the AOI's boundary. Three
  orderings of the same points:
  - **hilbert:** Hilbert order on the global pixel grid; row groups are compact areas holding every date.
  - **blocks:** Hilbert-ordered blocks of about `INDEX_DAYS` row groups, each sorted newest date first;
    dense areas get one row group per date (date statistics prune), sparse areas share row groups.
  - **by-date:** newest date first, Hilbert within each date; a row group never spans two dates.

All methods return identical rows (checked below). Run it on a notebook server in us-east-1, the data's
region; the index build starts a Coiled cluster.

In [ ]:
import time
import tracemalloc
from concurrent.futures import ThreadPoolExecutor

import fsspec
import geopandas as gpd
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.fs as pafs
import pyarrow.parquet as pq
import s3fs
import shapely
import xarray as xr
from dask.distributed import as_completed

In [ ]:
# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
ANALYTICS_BUCKET = "lcl-analytics"
S3_OPTS = {"requester_pays": True}
# The alerts release the API serves; pin a version string to reproduce a run.
ALERTS_VERSION = fsspec.open(
    f"s3://{ANALYTICS_BUCKET}/zonal-statistics/integrated-alerts/latest", "rt", **S3_OPTS
).open().read().strip()
ALERTS_URI = f"s3://{ANALYTICS_BUCKET}/zarr/gfw_integrated_dist_alerts/{ALERTS_VERSION}/date_conf.zarr"
NATURAL_LANDS_URI = f"s3://{ANALYTICS_BUCKET}/zarr/sbtn-natural-lands/sbtn_natural_lands_all_classes.zarr"
GADM_URI = f"s3://{ANALYTICS_BUCKET}/vectors/aois/gadm/v4.1.85/aois.parquet"
WDPA_URI = f"s3://{ANALYTICS_BUCKET}/vectors/wdpa.parquet"

# Alert encoding (as in the API): alert_date = days since 2014-12-31, confidence 2/3/4.
ALERT_EPOCH = pd.Timestamp("2014-12-31")
CONFIDENCE = {2: "low", 3: "high", 4: "highest"}
# Keep in sync with LAND_FILTER_CLASSES in api/app/analysis/common/analysis.py
LAND_FILTER_CLASSES = {"natural_lands": list(range(2, 12)), "natural_forests": [2, 5, 8, 9]}

# AOIs from small to large: (label, source, id); source is "gadm" (aoi_id) or "wdpa" (SITE_ID).
AOIS = [
    ("sparse municipality", "gadm", "BRA.12.1"),
    ("dense municipality", "gadm", "BRA.14.87"),
    ("state (Para)", "gadm", "BRA.14"),
    ("country (Brazil)", "gadm", "BRA"),
]
# Widget filters. Windows count back from the latest dataset update (not from today): the newest
# alert date in the release, which lags the release date by a few days.
WINDOW_DAYS = {"last day": 1, "last 3 days": 3, "last week": 7}
CONFIDENCE_FILTERS = {"all": [2, 3, 4], "high+": [3, 4]}
INTERSECTION_FILTERS = {"none": None, "natural lands": LAND_FILTER_CLASSES["natural_lands"]}
PAGE_SIZE = 1000

RASTER_THREADS = 16             # tiles read in parallel by the raster method
RASTER_MAX_TILES = 50           # larger AOIs: one raster scan (last week, no filters); it costs the same for any filter
N_REPEATS = 3                   # vector queries are repeated; each raster scan runs once

INDEX_DAYS = max(WINDOW_DAYS.values())
TILE_PX = 10_000                # one alerts zarr chunk (1 x 1 degree at 10 m)
INDEX_ROW_GROUP = 10_000        # small groups (~50 KB): statistics prune close to the AOI
BLOCK_ROWS = INDEX_DAYS * INDEX_ROW_GROUP  # "blocks" layout: Hilbert runs of this many points, newest date first inside
HILBERT_ORDER = 22              # 2**22 px covers the 3.6M-column global 10 m grid
LAYOUTS = ["hilbert", "blocks", "by-date"]

INDEX_SCOPE = None              # None: global. Or a list of AOI ids: index only their tiles (quick tests)
REBUILD_INDEX = False           # False: reuse this release's index if it exists
BUILD_CLUSTER = "coiled"        # "coiled", or "local" (in-process, for small INDEX_SCOPE tests)
INDEX_PREFIX = (f"s3://{ANALYTICS_BUCKET}/zonal-statistics/integrated-alerts/test/alert-points-index/"
                f"{ALERTS_VERSION}/{'global' if INDEX_SCOPE is None else '-'.join(INDEX_SCOPE)}")
INDEX_URIS = {layout: f"{INDEX_PREFIX}/{layout}.parquet" for layout in LAYOUTS}
print(f"alerts {ALERTS_VERSION}; index -> {INDEX_PREFIX}")

In [ ]:
# ---------------------------------------------------------------------------
# Shared setup (not charged to any method): zarrs, AOIs and the tiles they touch
# ---------------------------------------------------------------------------
t0 = time.perf_counter()
fs = s3fs.S3FileSystem(**S3_OPTS)
# Native reader for index queries, with retries and timeouts as an API would use.
s3 = pafs.S3FileSystem(region=pafs.resolve_s3_region(ANALYTICS_BUCKET), connect_timeout=5, request_timeout=30,
                       retry_strategy=pafs.AwsStandardS3RetryStrategy(max_attempts=5))


def open_zarrs():
    """Alerts and natural lands zarrs with their pixel-center coordinates."""
    alerts = xr.open_zarr(ALERTS_URI, storage_options=S3_OPTS).squeeze("band", drop=True)
    nl = xr.open_zarr(NATURAL_LANDS_URI, storage_options=S3_OPTS)["band_data"].squeeze("band", drop=True)
    return {"alerts": alerts, "nl": nl, "x": alerts.x.values, "y": alerts.y.values,
            "nl_x": nl.x.values, "nl_y": nl.y.values, "nl_chunks": nl.data.chunksize}


ZARRS = open_zarrs()
x_centers, y_centers = ZARRS["x"], ZARRS["y"]                     # 10 m pixel centers
HALF = abs(float(x_centers[1] - x_centers[0])) / 2


def tile_window(x, y, ti, tj):
    """Row/col ranges and lon/lat bbox (pixel edges) of alerts tile (ti, tj)."""
    r0, r1 = ti * TILE_PX, min((ti + 1) * TILE_PX, len(y))
    c0, c1 = tj * TILE_PX, min((tj + 1) * TILE_PX, len(x))
    half = abs(float(x[1] - x[0])) / 2
    return (r0, r1), (c0, c1), (x[c0] - half, y[r1 - 1] - half, x[c1 - 1] + half, y[r0] + half)


def load_aoi(source, aoi_id):
    if source == "gadm":
        return gpd.read_parquet(GADM_URI, columns=["aoi_id", "geometry"], storage_options=S3_OPTS,
                                filters=pc.field("aoi_id") == aoi_id).geometry.iloc[0]
    return gpd.read_parquet(WDPA_URI, columns=["SITE_ID", "geometry"], storage_options=S3_OPTS,
                            filters=pc.field("SITE_ID") == aoi_id).geometry.union_all()


def aoi_context(geom):
    """The AOI's bbox as global row/col ranges, and the AOI clipped to each alerts tile it touches."""
    minx, miny, maxx, maxy = geom.bounds
    rows = (int(np.searchsorted(-y_centers, -maxy)), int(np.searchsorted(-y_centers, -miny)) - 1)
    cols = (int(np.searchsorted(x_centers, minx)), int(np.searchsorted(x_centers, maxx)) - 1)
    tiles = {}
    for ti in range(rows[0] // TILE_PX, rows[1] // TILE_PX + 1):
        for tj in range(cols[0] // TILE_PX, cols[1] // TILE_PX + 1):
            clip = shapely.clip_by_rect(geom, *tile_window(x_centers, y_centers, ti, tj)[2])
            if not clip.is_empty:
                shapely.prepare(clip)
                tiles[(ti, tj)] = clip
    shapely.prepare(geom)
    return {"geom": geom, "rows": rows, "cols": cols, "tiles": tiles}


aois = {}
for label, source, aoi_id in AOIS:
    aois[label] = aoi_context(load_aoi(source, aoi_id))
    aois[label]["id"] = aoi_id
    print(f"{label} ({aoi_id}): {len(aois[label]['tiles'])} tiles")
print(f"setup {time.perf_counter() - t0:.0f}s")

In [ ]:
# ---------------------------------------------------------------------------
# Per-tile read, shared by the raster method (threads here) and the index build (Dask workers)
# ---------------------------------------------------------------------------
POINT_COLUMNS = ["row", "col", "alert_date", "confidence", "natural_lands_class"]


def _empty_points():
    return pd.DataFrame({"row": np.empty(0, np.int32), "col": np.empty(0, np.int32),
                         "alert_date": np.empty(0, np.int16), "confidence": np.empty(0, np.int8),
                         "natural_lands_class": np.empty(0, np.uint8)})


def read_tile(zarrs, ti, tj, day_lo, day_hi, confidences, classes=None, clip=None):
    """Alert pixels of one tile in [day_lo, day_hi] at the given confidences, with their natural lands
    class; optionally only classes in `classes` and only pixel centers inside `clip`.
    Returns (points, natural lands window read or None)."""
    x, y = zarrs["x"], zarrs["y"]
    (r0, r1), (c0, c1), (west, south, east, north) = tile_window(x, y, ti, tj)
    # scheduler="synchronous" loads inside this thread/task (no nested graph on a worker); passed per
    # call because dask.config.set is global and another thread could reset it mid-task.
    a = zarrs["alerts"][["alert_date", "confidence"]].isel(y=slice(r0, r1), x=slice(c0, c1)).load(scheduler="synchronous")
    date, conf = a["alert_date"].values, a["confidence"].values
    rows, cols = np.nonzero((date >= day_lo) & (date <= day_hi) & np.isin(conf, confidences))
    if len(rows) == 0:
        return _empty_points(), None
    lon, lat = x[c0 + cols], y[r0 + rows]

    # Natural lands class: each 10 m pixel takes the 30 m pixel its center is nearest to, within half a
    # 30 m pixel (0 outside the layer), exactly as reindex_like does in the pipeline and the API.
    nl_x, nl_y = zarrs["nl_x"], zarrs["nl_y"]
    nl_res = abs(float(nl_x[1] - nl_x[0]))
    i0, i1 = np.searchsorted(-nl_y, -(north + nl_res)), np.searchsorted(-nl_y, -(south - nl_res))
    j0, j1 = np.searchsorted(nl_x, west - nl_res), np.searchsorted(nl_x, east + nl_res)
    nl_class = np.zeros(len(rows), dtype=np.uint8)
    if i1 > i0 and j1 > j0:
        win = zarrs["nl"].isel(y=slice(i0, i1), x=slice(j0, j1)).load(scheduler="synchronous")
        yi = win.indexes["y"].get_indexer(lat, method="nearest", tolerance=nl_res / 2)
        xi = win.indexes["x"].get_indexer(lon, method="nearest", tolerance=nl_res / 2)
        found = (yi >= 0) & (xi >= 0)
        nl_class[found] = win.values[yi[found], xi[found]]

    keep = np.ones(len(rows), dtype=bool)
    if classes is not None:
        keep &= np.isin(nl_class, classes)
    if clip is not None:
        keep[keep] = shapely.contains_xy(clip, lon[keep], lat[keep])
    return pd.DataFrame({
        "row": (r0 + rows[keep]).astype(np.int32), "col": (c0 + cols[keep]).astype(np.int32),
        "alert_date": date[rows[keep], cols[keep]].astype(np.int16),
        "confidence": conf[rows[keep], cols[keep]].astype(np.int8),
        "natural_lands_class": nl_class[keep],
    }), (i0, i1, j0, j1)


def newest_first(df):
    return df.sort_values(["alert_date", "row", "col"], ascending=[False, True, True], ignore_index=True)


def timed(fn, *args, **kwargs):
    """Run fn, returning (result, seconds, peak traced memory in GB)."""
    tracemalloc.start()
    t = time.perf_counter()
    result = fn(*args, **kwargs)
    seconds = time.perf_counter() - t
    peak = tracemalloc.get_traced_memory()[1] / 1e9
    tracemalloc.stop()
    return result, seconds, peak

In [ ]:
# ---------------------------------------------------------------------------
# Build the global index (one-off per release) on a Dask cluster, in three layouts
# ---------------------------------------------------------------------------
# The latest dataset update: the newest alert date in this release, read from the admin table's
# parquet row-group statistics (no data read). Every window ends there.
_admin = pq.ParquetFile(fs.open(f"{ANALYTICS_BUCKET}/zonal-statistics/integrated-alerts/"
                                f"{ALERTS_VERSION}/admin-integrated-alerts.parquet")).metadata
_date_col = [_admin.schema.column(j).name for j in range(_admin.num_columns)].index("alert_date")
LATEST_DATE = max(_admin.row_group(g).column(_date_col).statistics.max for g in range(_admin.num_row_groups))
LATEST_DAY = (pd.Timestamp(LATEST_DATE) - ALERT_EPOCH).days


def _worker_zarrs():
    """Open the zarrs once per worker. The cache lives on the worker object, never in a notebook
    global: cloudpickle ships a notebook function's globals with it, and an opened zarr in a global
    made every task ~50 MB."""
    from distributed import get_worker
    cache = get_worker().__dict__
    if "_alert_points_zarrs" not in cache:
        cache["_alert_points_zarrs"] = open_zarrs()
    return cache["_alert_points_zarrs"]


def index_tile(task):
    ti, tj, day_lo, day_hi = task
    return read_tile(_worker_zarrs(), ti, tj, day_lo, day_hi, [2, 3, 4])[0]


def index_tiles():
    """Alerts tiles that exist in the zarr (chunks never written hold no alerts)."""
    prefix = f"{ALERTS_URI.removeprefix('s3://')}/alert_date/c/0/"
    found = sorted({tuple(int(p) for p in k[len(prefix):].split("/")[:2]) for k in fs.find(prefix)})
    if INDEX_SCOPE is not None:
        scope = {key for a in aois.values() if a["id"] in INDEX_SCOPE for key in a["tiles"]}
        found = [t for t in found if t in scope]
    return found


def start_cluster():
    if BUILD_CLUSTER == "local":
        from distributed import LocalCluster
        cluster = LocalCluster(n_workers=1, threads_per_worker=8, processes=False, dashboard_address=None)
        return cluster, cluster.get_client()
    import coiled
    cluster = coiled.Cluster(
        name="alert_points_index",
        region="us-east-1",
        n_workers=10,
        tags={"project": "gnw_zonal_stat"},
        scheduler_vm_types=["r7i.xlarge"],
        worker_vm_types=["r7i.2xlarge"],
        compute_purchase_option="on-demand",
        no_client_timeout="10 minutes",
        worker_options={"nthreads": 4},
    )
    cluster.adapt(minimum=10, maximum=40)
    return cluster, cluster.get_client()


def collect_points(client, tiles):
    """One task per tile (just its indices and the window); results come back in batches."""
    tasks = [(ti, tj, LATEST_DAY - INDEX_DAYS + 1, LATEST_DAY) for ti, tj in tiles]
    futures = client.map(index_tile, tasks, pure=False)
    parts, failed, done = [], [], 0
    for batch in as_completed(futures, with_results=True, raise_errors=False).batches():
        for fut, res in batch:
            if fut.status == "error":
                failed.append((fut.key, repr(fut.exception())))
            elif len(res):
                parts.append(res)
        done += len(batch)
        if done % 500 < len(batch) or done == len(futures):
            print(f"{done:,}/{len(futures):,} tiles, {sum(len(p) for p in parts):,} points, "
                  f"{len(failed)} failed", flush=True)
    if failed:
        raise RuntimeError(f"{len(failed)} tiles failed, e.g. {failed[:3]}")
    return pd.concat(parts, ignore_index=True)


def hilbert_index(x, y, order=HILBERT_ORDER):
    """Distance along a Hilbert curve over a 2**order x 2**order grid (vectorized xy -> d)."""
    x, y = x.astype(np.int64), y.astype(np.int64)
    n, d, s = 1 << order, np.zeros(len(x), dtype=np.int64), 1 << (order - 1)
    while s > 0:
        rx, ry = (x & s) > 0, (y & s) > 0
        d += s * s * ((3 * rx) ^ ry)
        flip = ~ry & rx
        x, y = np.where(flip, n - 1 - x, x), np.where(flip, n - 1 - y, y)
        x, y = np.where(~ry, y, x), np.where(~ry, x, y)  # swap when ry == 0
        s >>= 1
    return d


def _runs(*keys):
    """Start/stop offsets of the runs of equal key tuples in sorted arrays."""
    change = np.zeros(len(keys[0]), dtype=bool)
    change[0] = True
    for k in keys:
        change[1:] |= k[1:] != k[:-1]
    starts = np.flatnonzero(change)
    return list(zip(starts.tolist(), np.append(starts[1:], len(keys[0])).tolist()))


def write_layout(uri, table, slices, min_rows=0):
    """Write `table` to one parquet file. Each slice (start, stop) starts a new row group (split at
    INDEX_ROW_GROUP rows); consecutive slices are merged until they hold at least `min_rows`."""
    with fsspec.open(uri, "wb", **S3_OPTS) as f, pq.ParquetWriter(f, table.schema, write_statistics=True) as w:
        pending = []
        for start, stop in slices:
            pending.append(table.slice(start, stop - start))
            if sum(len(t) for t in pending) >= min_rows:
                w.write_table(pa.concat_tables(pending), row_group_size=INDEX_ROW_GROUP)
                pending = []
        if pending:
            w.write_table(pa.concat_tables(pending), row_group_size=INDEX_ROW_GROUP)


def build_index():
    tiles = index_tiles()
    print(f"{len(tiles):,} alerts tiles; window {LATEST_DAY - INDEX_DAYS + 1}..{LATEST_DAY}", flush=True)
    cluster, client = start_cluster()
    try:
        points = collect_points(client, tiles)
    finally:
        client.close()
        cluster.close()
    h = hilbert_index(points["col"].to_numpy(), points["row"].to_numpy())
    newest = -points["alert_date"].to_numpy().astype(np.int32)

    # hilbert: one Hilbert run, cut into row groups of INDEX_ROW_GROUP.
    order = np.argsort(h, kind="stable")
    table = pa.Table.from_pandas(points.iloc[order], preserve_index=False)
    write_layout(INDEX_URIS["hilbert"], table, [(0, len(table))])
    # blocks: consecutive Hilbert runs of BLOCK_ROWS points, newest date first inside each; one row group
    # per (block, date) where dense, merged with the next slices where sparse.
    block = np.empty(len(h), dtype=np.int64)
    block[order] = np.arange(len(h)) // BLOCK_ROWS
    order = np.lexsort((h, newest, block))
    table = pa.Table.from_pandas(points.iloc[order], preserve_index=False)
    write_layout(INDEX_URIS["blocks"], table, _runs(block[order], newest[order]), min_rows=INDEX_ROW_GROUP // 2)
    # by-date: newest date first, Hilbert within each date; no row group spans two dates.
    order = np.lexsort((h, newest))
    table = pa.Table.from_pandas(points.iloc[order], preserve_index=False)
    write_layout(INDEX_URIS["by-date"], table, _runs(newest[order]))
    return len(tiles), len(points)


if REBUILD_INDEX or not all(fs.exists(uri) for uri in INDEX_URIS.values()):
    (n_tiles, n_points), build_s, _ = timed(build_index)
    print(f"built {n_points:,} points from {n_tiles:,} tiles in {build_s / 60:.1f} min")
else:
    print("reusing the existing index for", ALERTS_VERSION)
print("latest dataset update", LATEST_DATE)

In [ ]:
# ---------------------------------------------------------------------------
# Index footers: read once and cached, as an API process would; statistics as arrays
# ---------------------------------------------------------------------------
def load_index(uri):
    meta = pq.ParquetFile(fs.open(uri)).metadata
    names = [meta.schema.column(j).name for j in range(meta.num_columns)]
    groups = [meta.row_group(g) for g in range(meta.num_row_groups)]

    def stat(col, attr):
        j = names.index(col)
        return np.array([getattr(rg.column(j).statistics, attr) for rg in groups])

    spans = []
    for rg in groups:
        cols = [rg.column(j) for j in range(rg.num_columns)]
        start = min(c.dictionary_page_offset if c.has_dictionary_page else c.data_page_offset for c in cols)
        spans.append((start, start + sum(c.total_compressed_size for c in cols)))
    ix = {"path": uri.removeprefix("s3://"), "meta": meta, "num_rows": np.array([rg.num_rows for rg in groups]),
          "spans": np.array(spans, dtype=np.int64), "bytes": fs.size(uri)}
    for col, key in (("row", "r"), ("col", "c"), ("alert_date", "d")):
        ix[key + "min"], ix[key + "max"] = stat(col, "min"), stat(col, "max")
    # Each row group's extent as a box (pixel edges), for testing against the AOI polygon.
    ix["boxes"] = shapely.box(x_centers[ix["cmin"]] - HALF, y_centers[ix["rmax"]] - HALF,
                              x_centers[ix["cmax"]] + HALF, y_centers[ix["rmin"]] + HALF)
    return ix


INDEX = {layout: load_index(uri) for layout, uri in INDEX_URIS.items()}
index_table = pd.DataFrame({
    layout: {"points": ix["meta"].num_rows, "row groups": ix["meta"].num_row_groups,
             "MB": ix["bytes"] / 1e6, "bytes/point": ix["bytes"] / max(ix["meta"].num_rows, 1),
             "footer KB": ix["meta"].serialized_size / 1e3,
             "row groups spanning >1 date": int((ix["dmax"] > ix["dmin"]).sum())}
    for layout, ix in INDEX.items()}).T
index_table

In [ ]:
# ---------------------------------------------------------------------------
# Raster method: scan the AOI's chunks for this request
# ---------------------------------------------------------------------------
NL_CHUNKS = ZARRS["nl_chunks"]  # (y, x) pixels per natural lands chunk
ALERTS_KEY = ALERTS_URI.removeprefix("s3://")
NL_KEY = NATURAL_LANDS_URI.removeprefix("s3://")
_sizes = {}


def object_bytes(keys):
    """Compressed bytes of the zarr chunk objects read (S3 object sizes, cached)."""
    for k in set(keys) - set(_sizes):
        _sizes[k] = fs.size(k) if fs.exists(k) else 0
    return sum(_sizes[k] for k in set(keys))


def window_days(days):
    return LATEST_DAY - days + 1, LATEST_DAY


def raster_query(aoi, days, confidences, classes):
    """Full result, newest first, plus the chunk keys read. The first page needs the same full scan,
    since the newest alerts can be in any tile."""
    lo, hi = window_days(days)
    items = list(aoi["tiles"].items())
    with ThreadPoolExecutor(RASTER_THREADS) as pool:
        parts = list(pool.map(lambda kv: read_tile(ZARRS, *kv[0], lo, hi, confidences, classes, kv[1]), items))
    keys = []
    for ((ti, tj), _), (_, nl_window) in zip(items, parts):
        keys += [f"{ALERTS_KEY}/alert_date/c/0/{ti}/{tj}", f"{ALERTS_KEY}/confidence/c/0/{ti}/{tj}"]
        if nl_window is not None:
            i0, i1, j0, j1 = nl_window
            keys += [f"{NL_KEY}/band_data/c/0/{r}/{c}"
                     for r in range(i0 // NL_CHUNKS[0], (i1 - 1) // NL_CHUNKS[0] + 1)
                     for c in range(j0 // NL_CHUNKS[1], (j1 - 1) // NL_CHUNKS[1] + 1)]
    return newest_first(pd.concat([p for p, _ in parts], ignore_index=True)), keys

In [ ]:
# ---------------------------------------------------------------------------
# Vector method: row groups selected from cached statistics and the AOI polygon, one read per step
# ---------------------------------------------------------------------------
def select_groups(ix, aoi, day_lo, day_hi):
    """Row groups whose statistics overlap the window and the AOI's bbox (vectorized) and whose box
    intersects the AOI polygon. Returns (group ids, interior flags): an interior group lies entirely
    inside the AOI, so its points skip the point-in-polygon test."""
    (r0, r1), (c0, c1) = aoi["rows"], aoi["cols"]
    cand = np.flatnonzero((ix["rmax"] >= r0) & (ix["rmin"] <= r1) & (ix["cmax"] >= c0) & (ix["cmin"] <= c1)
                          & (ix["dmax"] >= day_lo) & (ix["dmin"] <= day_hi))
    boxes = ix["boxes"][cand]
    hit = shapely.intersects(aoi["geom"], boxes)
    return cand[hit], shapely.contains_properly(aoi["geom"], boxes[hit])


def byte_ranges(ix, groups, hole=8192):
    """Bytes and merged byte ranges of the row groups (gaps under `hole` merged, as arrow's
    pre-buffering does): about the number of GET requests the read makes."""
    spans = ix["spans"][np.sort(groups)]
    merged = 1 + int((spans[1:, 0] - spans[:-1, 1] > hole).sum()) if len(spans) else 0
    return int((spans[:, 1] - spans[:, 0]).sum()), merged


def keep_rows(aoi, pts, interior, day_lo, day_hi, confidences, classes):
    """Window/confidence/intersection filters, then contains_xy for points from boundary row groups,
    per tile against the clipped AOI (the raster method's test)."""
    keep = (pts["alert_date"].between(day_lo, day_hi) & pts["confidence"].isin(confidences)
            & pts["row"].between(*aoi["rows"]) & pts["col"].between(*aoi["cols"])).to_numpy()
    if classes is not None:
        keep &= pts["natural_lands_class"].isin(classes).to_numpy()
    pts, inside = pts[keep], interior[keep].copy()
    edge = np.flatnonzero(~inside)
    rows_, cols_ = pts["row"].to_numpy()[edge], pts["col"].to_numpy()[edge]
    tile_id = (rows_ // TILE_PX).astype(np.int64) * 1000 + cols_ // TILE_PX
    order = np.argsort(tile_id, kind="stable")
    for idx in np.split(order, np.flatnonzero(np.diff(tile_id[order])) + 1):
        if len(idx) == 0:
            continue
        clip = aoi["tiles"].get((int(tile_id[idx[0]] // 1000), int(tile_id[idx[0]] % 1000)))
        if clip is not None:
            inside[edge[idx]] = shapely.contains_xy(clip, x_centers[cols_[idx]], y_centers[rows_[idx]])
    return pts[inside]


def vector_query(layout, aoi, days, confidences, classes, page_size=None):
    """Full result: one read of the window's row groups under the AOI. First page: newest date first,
    reading each date's not-yet-read row groups and stopping once the page is complete.
    Returns (rows newest first, bytes fetched, requests)."""
    ix = INDEX[layout]
    lo, hi = window_days(days)
    steps = [(lo, hi)] if page_size is None else [(d, d) for d in range(hi, lo - 1, -1)]
    found, seen, fetched, requests = [], np.empty(0, dtype=np.int64), 0, 0
    for day_lo, day_hi in steps:
        groups, interior = select_groups(ix, aoi, day_lo, day_hi)
        new = ~np.isin(groups, seen)
        groups, interior = groups[new], interior[new]
        if len(groups):
            seen = np.concatenate([seen, groups])
            pf = pq.ParquetFile(s3.open_input_file(ix["path"]), metadata=ix["meta"], pre_buffer=True)
            pts = pf.read_row_groups(groups.tolist()).to_pandas()
            found.append(keep_rows(aoi, pts, np.repeat(interior, ix["num_rows"][groups]),
                                   lo, hi, confidences, classes))
            b, r = byte_ranges(ix, groups)
            fetched, requests = fetched + b, requests + r
        # Every row group holding dates >= day_lo has been read: the rows from day_lo on are complete.
        if page_size is not None and sum(int((f["alert_date"] >= day_lo).sum()) for f in found) >= page_size:
            break
    result = newest_first(pd.concat(found, ignore_index=True)) if found else _empty_points()
    return (result.head(page_size) if page_size else result), fetched, requests

In [ ]:
# ---------------------------------------------------------------------------
# Benchmark: AOI x window x confidence x intersection; raster and each index layout
# ---------------------------------------------------------------------------
# Fairness: the layouts run in a different order for each combination, repeats are interleaved
# (layout A, B, C, A, B, C, ...) rather than back to back, and each timed query is preceded by a tiny
# untimed read of the same file so no layout pays for a connection dropped during the raster scan.
def same_rows(a, b):
    return a[POINT_COLUMNS].reset_index(drop=True).equals(b[POINT_COLUMNS].reset_index(drop=True))


def warm_up(layout):
    with s3.open_input_file(INDEX[layout]["path"]) as f:
        f.read_at(1, 0)


records, problems, samples = [], [], {}
combo_no = 0
for label, aoi in aois.items():
    big = len(aoi["tiles"]) > RASTER_MAX_TILES
    for wname, days in WINDOW_DAYS.items():
        for cname, confidences in CONFIDENCE_FILTERS.items():
            for iname, classes in INTERSECTION_FILTERS.items():
                combo = (label, wname, cname, iname)
                base = dict(aoi=label, window=wname, confidence=cname, intersection=iname)
                expected = None
                if not big or (wname, cname, iname) == ("last week", "all", "none"):
                    (expected, keys), secs, gb = timed(raster_query, aoi, days, confidences, classes)
                    mb, nreq = object_bytes(keys) / 1e6, len(set(keys))
                    for mode in ("first page", "full"):
                        records.append(dict(base, method="raster", mode=mode, seconds=secs, mb_read=mb,
                                            requests=nreq, peak_gb=gb,
                                            rows=len(expected) if mode == "full" else min(PAGE_SIZE, len(expected))))
                order = LAYOUTS[combo_no % len(LAYOUTS):] + LAYOUTS[:combo_no % len(LAYOUTS)]
                combo_no += 1
                outputs = {}
                for _ in range(N_REPEATS):
                    for layout in order:
                        for mode, page in (("first page", PAGE_SIZE), ("full", None)):
                            warm_up(layout)
                            (out, fetched, nreq), secs, gb = timed(vector_query, layout, aoi, days, confidences, classes, page)
                            records.append(dict(base, method=layout, mode=mode, seconds=secs, mb_read=fetched / 1e6,
                                                requests=nreq, peak_gb=gb, rows=len(out)))
                            outputs[(layout, mode)] = out
                if expected is None:  # no raster scan for this combination: check the layouts agree
                    expected = outputs[(LAYOUTS[0], "full")]
                for layout in LAYOUTS:
                    if not same_rows(outputs[(layout, "full")], expected):
                        problems.append(f"{combo} {layout}: full results differ "
                                        f"({len(expected):,} vs {len(outputs[(layout, 'full')]):,})")
                    if not same_rows(outputs[(layout, "first page")], expected.head(PAGE_SIZE)):
                        problems.append(f"{combo} {layout}: first page differs")
                if (wname, cname, iname) == ("last week", "all", "none"):
                    samples[label] = expected.head(10)
                done = pd.DataFrame([r for r in records if r["aoi"] == label and r["window"] == wname
                                     and r["confidence"] == cname and r["intersection"] == iname and r["mode"] == "full"])
                stats = done.groupby("method").agg(s=("seconds", "median"), req=("requests", "median"), mb=("mb_read", "median"))
                print(f"{combo} [full, median]: " + " | ".join(
                    f"{m} {stats.loc[m, 's']:.2f}s/{stats.loc[m, 'req']:.0f} req/{stats.loc[m, 'mb']:.2f} MB"
                    for m in ["raster"] + LAYOUTS if m in stats.index) + f"; {len(expected):,} alerts", flush=True)
results = pd.DataFrame(records)
print("all combinations identical" if not problems else "\n".join(problems))

In [ ]:
# ---------------------------------------------------------------------------
# Results
# ---------------------------------------------------------------------------
summary = (results.groupby(["aoi", "window", "confidence", "intersection", "mode", "method"], sort=False)
           .agg(median_s=("seconds", "median"), mb_read=("mb_read", "median"), requests=("requests", "median"),
                peak_gb=("peak_gb", "max"), rows=("rows", "max"))
           .unstack("method"))
print(f"alerts {ALERTS_VERSION}, latest dataset update {LATEST_DATE}, page size {PAGE_SIZE}")
print(index_table.round(2).to_string(), "\n")
pd.set_option("display.width", 250)
print(summary[["median_s", "requests", "mb_read"]].round(3).to_string())

In [ ]:
# What a widget row looks like (lat/lon are the pixel centers)
sample = samples["dense municipality"] if "dense municipality" in samples else next(iter(samples.values()))
pd.DataFrame({
    "lat": y_centers[sample["row"]], "lon": x_centers[sample["col"]],
    "alert_date": ALERT_EPOCH + pd.to_timedelta(sample["alert_date"].astype(int), unit="D"),
    "confidence": sample["confidence"].map(CONFIDENCE),
    "natural_lands_class": sample["natural_lands_class"],
    "is_new": sample["alert_date"] > LATEST_DAY - 7,
})

## Reading the results

- **Raster** cost is about the same for every filter and for the first page: every request reads all the
  AOI's alerts chunks, and the newest page is only known after the whole scan. AOIs with more than
  `RASTER_MAX_TILES` tiles get one scan (last week, no filters).
- **Vector** cost is set by the row groups read (`requests` are merged byte ranges, `mb_read` their size):
  - **hilbert** reads every date in the AOI's row groups, whatever the window; few, contiguous ranges.
  - **blocks** keeps the AOI's data together and lets date statistics skip other dates in dense areas.
  - **by-date** reads only the window's dates, as separate per-date ranges fetched concurrently.
- For large AOIs most row groups are **interior** (their box lies inside the AOI), so only boundary row
  groups pay for the point-in-polygon test; a first page reads the newest date first and stops when full.
- `index_table` gives the global index size (`bytes/point`, `footer KB`): the footer is read once per
  process and release; `row groups spanning >1 date` shows where date statistics cannot prune.
- Peak memory is Python/numpy memory traced during the call (native buffers outside numpy are not).